[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/080.%20The%20International%20Freight%20Mode%20Selection%20Problem%20%28Air%20vs.%20Ocean%29/P80-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 7: Human-AI Symbiotic Partnership (The Centaur Model)

### Key assumptions
- Human experts and AI systems collaborate as equal partners in decision-making
- AI provides data-driven insights while humans contribute contextual wisdom
- Trust develops through transparent explanations and consistent performance
- The system learns from human feedback and adapts to individual preferences
- Decision quality improves through the synergy of human intuition and AI analytics

### Approach (step-by-step)
1. **Establish partnership framework**: Define roles and responsibilities for human-AI collaboration
2. **Create explainable AI**: Develop interpretable models with transparent reasoning processes
3. **Implement trust scoring**: Build confidence metrics based on prediction accuracy and human agreement
4. **Design feedback loops**: Enable humans to provide corrections and preferences to AI system
5. **Develop learning mechanisms**: AI adapts based on human feedback and performance patterns
6. **Create decision interface**: Build collaborative tools for joint decision-making
7. **Measure partnership effectiveness**: Track improvement metrics and satisfaction scores

### What to look for in the results
- Gradual improvement in decision quality as partnership matures
- Increasing trust scores and agreement rates between human and AI recommendations
- Learning curves showing AI adaptation to human preferences
- Decision transparency with explainable AI reasoning
- Performance comparison between pure AI, pure human, and symbiotic approaches

### Concrete example (from the source)
GlobalLogistics Partners implements Human-AI Symbiotic Partnership for freight mode selection:
- **Human Team**: 3 logistics experts with 15+ years experience each
- **AI System**: Explainable freight optimization model with confidence scoring
- **Partnership Duration**: 6-month pilot with weekly decision reviews
- **Decision Scope**: 847 freight mode decisions across 47 trade lanes

Results after 6 months: 94% human-AI agreement rate, 27% cost reduction vs pure human decisions, 18% improvement in on-time delivery vs pure AI decisions. The partnership identified 23 "edge cases" where human intuition corrected AI blind spots, and AI discovered 15 optimization patterns that humans had missed. Trust scores increased from 0.62 to 0.91, and decision time reduced by 40% while maintaining higher quality outcomes.

In [ ]:
# Import required packages for human-AI symbiotic partnership
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Shipment:
    """Data class for shipment information"""
    id: int
    weight: float  # kg
    value: float   # USD
    destination: str
    due_date: int  # days from now
    priority: str  # high, medium, low
    special_requirements: List[str]  # e.g., ['temperature_controlled', 'hazardous']

@dataclass
class TransportParams:
    """Parameters for transportation costs and times"""
    costs: Dict[str, Dict[str, float]]  # mode -> destination -> cost per kg
    transit_times: Dict[str, Dict[str, int]]  # mode -> destination -> days
    holding_rate: float  # daily holding cost rate as fraction of value
    late_penalty: float  # penalty per day late
    air_capacity: float  # monthly air freight capacity in kg

@dataclass
class HumanExpert:
    """Human logistics expert with experience and preferences"""
    expert_id: str
    name: str
    experience_years: int
    specialization: str  # e.g., 'air_freight', 'ocean_logistics', 'risk_management'
    risk_tolerance: float  # 0.0 (risk-averse) to 1.0 (risk-seeking)
    cost_sensitivity: float  # 0.0 (cost-insensitive) to 1.0 (cost-sensitive)
    reliability_score: float  # Historical decision accuracy
    confidence_threshold: float  # Minimum confidence to make decision

@dataclass
class AIRecommendation:
    """AI recommendation with explainable reasoning"""
    shipment_id: int
    recommended_mode: str
    confidence: float  # 0.0 to 1.0
    reasoning: List[str]  # Explainable reasons for recommendation
    cost_estimate: float
    risk_assessment: float
    alternative_modes: List[Dict]  # Alternative recommendations with scores

@dataclass
class HumanDecision:
    """Human expert decision with reasoning"""
    expert_id: str
    shipment_id: int
    chosen_mode: str
    reasoning: str
    confidence: float
    time_to_decide: int  # minutes
    consulted_ai: bool

@dataclass
class SymbioticDecision:
    """Joint human-AI decision with partnership metrics"""
    shipment_id: int
    final_mode: str
    human_confidence: float
    ai_confidence: float
    agreement: bool  # True if human and AI agreed
    decision_time: int  # minutes
    learning_outcome: str  # 'human_corrected', 'ai_corrected', 'mutual_agreement'
    trust_score: float

# Create comprehensive human-AI partnership environment
def create_partnership_environment() -> Tuple[List[Shipment], TransportParams, List[HumanExpert]]:
    """
    Create a comprehensive human-AI partnership environment
    """
    # Extended shipment set with special requirements
    shipments = [
        Shipment(1, 800, 1500000, "rotterdam", 8, "high", ["temperature_controlled"]),
        Shipment(2, 1200, 800000, "los_angeles", 15, "medium", []),
        Shipment(3, 3500, 600000, "rotterdam", 25, "low", ["hazardous"]),
        Shipment(4, 600, 2000000, "singapore", 10, "high", ["temperature_controlled", "high_value"]),
        Shipment(5, 2000, 400000, "hamburg", 20, "medium", []),
        Shipment(6, 1500, 900000, "new_york", 12, "high", []),
        Shipment(7, 4000, 300000, "rotterdam", 35, "low", ["bulk_cargo"]),
        Shipment(8, 900, 1200000, "los_angeles", 18, "medium", ["fragile"]),
        Shipment(9, 750, 1800000, "singapore", 7, "high", ["temperature_controlled", "pharmaceutical"]),
        Shipment(10, 2800, 750000, "hamburg", 22, "medium", ["oversized"])
    ]

    # Transportation parameters
    params = TransportParams(
        costs={
            "ocean": {
                "rotterdam": 2.2, "los_angeles": 1.9, "singapore": 2.8,
                "hamburg": 2.1, "new_york": 1.8
            },
            "air": {
                "rotterdam": 6.8, "los_angeles": 6.2, "singapore": 7.5,
                "hamburg": 6.5, "new_york": 5.9
            }
        },
        transit_times={
            "ocean": {
                "rotterdam": 28, "los_angeles": 20, "singapore": 35,
                "hamburg": 26, "new_york": 18
            },
            "air": {
                "rotterdam": 3, "los_angeles": 2, "singapore": 4,
                "hamburg": 3, "new_york": 2
            }
        },
        holding_rate=0.0012,
        late_penalty=1500,
        air_capacity=15000
    )

    # Human expert team
    experts = [
        HumanExpert("EXP001", "Sarah Chen", 18, "air_freight", 0.3, 0.7, 0.92, 0.75),
        HumanExpert("EXP002", "Marcus Rodriguez", 22, "ocean_logistics", 0.5, 0.8, 0.89, 0.80),
        HumanExpert("EXP003", "Aisha Patel", 15, "risk_management", 0.2, 0.6, 0.94, 0.85)
    ]

    return shipments, params, experts

# Initialize partnership environment
shipments, params, experts = create_partnership_environment()

print("Human-AI Partnership Environment Initialized:")
print(f"  Shipments: {len(shipments)}")
print(f"  Human Experts: {len(experts)}")
print(f"  Expert Specializations: {[e.specialization for e in experts]}")
print(f"  Average Experience: {np.mean([e.experience_years for e in experts]):.1f} years")

Human-AI Partnership Environment Initialized:
  Shipments: 10
  Human Experts: 3
  Expert Specializations: ['air_freight', 'ocean_logistics', 'risk_management']
  Average Experience: 18.3 years


In [ ]:
class ExplainableAI:
    """
    Explainable AI system for freight mode selection
    Provides transparent reasoning and confidence scoring
    """

    def __init__(self, params: TransportParams):
        self.params = params
        self.decision_history = []
        self.feedback_history = []
        self.confidence_calibration = 0.5  # Initial calibration

        # AI capability parameters
        self.data_quality_score = 0.85
        self.model_accuracy = 0.88
        self.reasoning_engine = "rule_based_with_ml"

    def analyze_shipment(self, shipment: Shipment) -> AIRecommendation:
        """
        Analyze shipment and provide explainable recommendation
        """
        # Calculate costs for both modes
        ocean_cost = self._calculate_total_cost(shipment, "ocean")
        air_cost = self._calculate_total_cost(shipment, "air")

        # Generate reasoning factors
        reasoning_factors = self._generate_reasoning(shipment, ocean_cost, air_cost)

        # Determine recommendation
        if shipment.priority == "high" and shipment.due_date <= 10:
            recommended_mode = "air"
            confidence = 0.85
        elif ocean_cost < air_cost * 0.7:
            recommended_mode = "ocean"
            confidence = 0.90
        elif "temperature_controlled" in shipment.special_requirements and shipment.value > 1000000:
            recommended_mode = "air"
            confidence = 0.80
        else:
            recommended_mode = "ocean"
            confidence = 0.75

        # Adjust confidence based on data quality and model accuracy
        confidence *= self.data_quality_score * self.model_accuracy
        confidence = min(0.95, confidence)  # Cap at 95%

        # Calculate risk assessment
        risk_assessment = self._calculate_risk_assessment(shipment, recommended_mode)

        # Generate alternatives
        alternatives = self._generate_alternatives(shipment, ocean_cost, air_cost, recommended_mode)

        recommendation = AIRecommendation(
            shipment_id=shipment.id,
            recommended_mode=recommended_mode,
            confidence=confidence,
            reasoning=reasoning_factors,
            cost_estimate=ocean_cost if recommended_mode == "ocean" else air_cost,
            risk_assessment=risk_assessment,
            alternative_modes=alternatives
        )

        self.decision_history.append(recommendation)
        return recommendation

    def _calculate_total_cost(self, shipment: Shipment, mode: str) -> float:
        """
        Calculate total cost for shipment mode
        """
        # Transportation cost
        transport_cost = shipment.weight * self.params.costs[mode][shipment.destination]

        # Transit time
        transit_time = self.params.transit_times[mode][shipment.destination]

        # Holding cost
        if transit_time < shipment.due_date:
            holding_cost = (shipment.due_date - transit_time) * self.params.holding_rate * shipment.value
        else:
            holding_cost = 0

        # Late penalty
        if transit_time > shipment.due_date:
            late_days = transit_time - shipment.due_date
            late_penalty_cost = late_days * self.params.late_penalty
        else:
            late_penalty_cost = 0

        # Special requirements cost
        special_cost = 0
        if "temperature_controlled" in shipment.special_requirements:
            special_cost += shipment.weight * 0.5  # $0.50 per kg for refrigeration
        if "hazardous" in shipment.special_requirements:
            special_cost += shipment.weight * 1.2  # $1.20 per kg for hazardous handling
        if "high_value" in shipment.special_requirements:
            special_cost += shipment.value * 0.001  # 0.1% insurance premium

        return transport_cost + holding_cost + late_penalty_cost + special_cost

    def _generate_reasoning(self, shipment: Shipment, ocean_cost: float, air_cost: float) -> List[str]:
        """
        Generate explainable reasoning for recommendation
        """
        reasoning = []

        # Cost-based reasoning
        cost_difference = air_cost - ocean_cost
        if cost_difference > 0:
            reasoning.append(f"Ocean freight saves ${cost_difference:.0f} vs air freight")
        else:
            reasoning.append(f"Air freight only ${abs(cost_difference):.0f} more than ocean")

        # Time-based reasoning
        ocean_time = self.params.transit_times["ocean"][shipment.destination]
        air_time = self.params.transit_times["air"][shipment.destination]
        time_savings = ocean_time - air_time

        if shipment.due_date <= ocean_time:
            reasoning.append(f"Ocean transit ({ocean_time} days) would miss due date (day {shipment.due_date})")
            reasoning.append(f"Air transit ({air_time} days) meets due date with {shipment.due_date - air_time} days buffer")
        elif time_savings > 20:
            reasoning.append(f"Air freight saves {time_savings} days transit time")

        # Priority-based reasoning
        if shipment.priority == "high":
            reasoning.append("High priority shipment favors faster transport")
        elif shipment.priority == "low" and ocean_cost < air_cost * 0.8:
            reasoning.append("Low priority allows cost optimization with ocean freight")

        # Special requirements reasoning
        if "temperature_controlled" in shipment.special_requirements:
            reasoning.append("Temperature-controlled cargo requires careful handling")
            if shipment.value > 1000000:
                reasoning.append("High-value temperature-sensitive cargo favors air freight")

        if "hazardous" in shipment.special_requirements:
            reasoning.append("Hazardous materials have stricter regulations for air transport")

        # Value-based reasoning
        if shipment.value > 1500000:
            reasoning.append(f"High-value cargo (${shipment.value:,}) warrants faster transport")

        return reasoning

    def _calculate_risk_assessment(self, shipment: Shipment, mode: str) -> float:
        """
        Calculate risk assessment for recommended mode
        """
        base_risk = 0.1  # Base risk level

        # Mode-specific risks
        if mode == "ocean":
            base_risk += 0.15  # Higher risk for ocean (weather, port delays)
            if shipment.destination in ["singapore", "rotterdam"]:
                base_risk += 0.05  # Higher congestion risk
        else:  # air
            base_risk += 0.05  # Lower base risk for air
            if "hazardous" in shipment.special_requirements:
                base_risk += 0.10  # Higher risk for hazardous air freight

        # Shipment-specific risks
        if "temperature_controlled" in shipment.special_requirements:
            base_risk += 0.03  # Risk of temperature excursions

        if shipment.due_date < 5:
            base_risk += 0.08  # Tight deadline increases risk

        return min(0.5, base_risk)  # Cap at 50% risk

    def _generate_alternatives(self, shipment: Shipment, ocean_cost: float,
                              air_cost: float, recommended: str) -> List[Dict]:
        """
        Generate alternative recommendations with scores
        """
        alternatives = []

        if recommended == "ocean":
            # Air as alternative
            air_score = max(0, 1 - (air_cost - ocean_cost) / ocean_cost)
            if shipment.priority == "high":
                air_score += 0.2  # Boost air score for high priority

            alternatives.append({
                "mode": "air",
                "score": min(1.0, air_score),
                "reason": "Faster transit for time-sensitive delivery"
            })
        else:  # recommended == "air"
            # Ocean as alternative
            ocean_score = max(0, 1 - (ocean_cost - air_cost) / air_cost)
            if shipment.priority == "low":
                ocean_score += 0.2  # Boost ocean score for low priority

            alternatives.append({
                "mode": "ocean",
                "score": min(1.0, ocean_score),
                "reason": "Cost savings for non-urgent shipment"
            })

        return alternatives

    def incorporate_feedback(self, shipment_id: int, human_decision: str,
                           human_reasoning: str, agreement: bool):
        """
        Incorporate human feedback to improve AI performance
        """
        feedback = {
            "shipment_id": shipment_id,
            "human_decision": human_decision,
            "human_reasoning": human_reasoning,
            "agreement": agreement,
            "timestamp": datetime.now()
        }

        self.feedback_history.append(feedback)

        # Adjust confidence calibration based on feedback
        if agreement:
            self.confidence_calibration = min(1.0, self.confidence_calibration + 0.02)
        else:
            self.confidence_calibration = max(0.3, self.confidence_calibration - 0.01)

        # Update model accuracy based on agreement rate
        recent_feedback = self.feedback_history[-20:]  # Last 20 decisions
        if len(recent_feedback) >= 5:
            agreement_rate = sum(1 for f in recent_feedback if f["agreement"]) / len(recent_feedback)
            self.model_accuracy = 0.5 + agreement_rate * 0.4  # Scale to 0.5-0.9 range

# Initialize explainable AI system
ai_system = ExplainableAI(params)
print("Explainable AI System initialized:")
print(f"  Data Quality Score: {ai_system.data_quality_score:.1%}")
print(f"  Model Accuracy: {ai_system.model_accuracy:.1%}")
print(f"  Reasoning Engine: {ai_system.reasoning_engine}")

Explainable AI System initialized:
  Data Quality Score: 85.0%
  Model Accuracy: 88.0%
  Reasoning Engine: rule_based_with_ml


In [ ]:
class HumanDecisionMaker:
    """
    Human expert decision-making with AI consultation
    """

    def __init__(self, expert: HumanExpert):
        self.expert = expert
        self.decision_history = []
        self.ai_consultation_count = 0
        self.ai_agreement_count = 0

        # Human decision patterns
        self.decision_patterns = {
            "high_priority_air_preference": 0.8,
            "cost_sensitivity": expert.cost_sensitivity,
            "risk_aversion": 1 - expert.risk_tolerance,
            "special_requirements_weight": 0.7
        }

    def make_decision(self, shipment: Shipment, ai_recommendation: Optional[AIRecommendation] = None) -> HumanDecision:
        """
        Make human decision, optionally consulting AI
        """
        start_time = time.time()

        # Decide whether to consult AI
        consulted_ai = False
        ai_influence = 0.0

        if ai_recommendation:
            # Consult AI if confidence is low or shipment is complex
            complexity_score = self._calculate_complexity(shipment)
            should_consult = (complexity_score > 0.6 or
                           self.expert.confidence_threshold < 0.7)

            if should_consult:
                consulted_ai = True
                self.ai_consultation_count += 1
                ai_influence = ai_recommendation.confidence * 0.3  # AI influence weight

        # Make decision based on expertise and AI input
        decision = self._make_reasoned_decision(shipment, ai_recommendation, ai_influence)

        decision_time = int((time.time() - start_time) * 60)  # Convert to minutes

        human_decision = HumanDecision(
            expert_id=self.expert.expert_id,
            shipment_id=shipment.id,
            chosen_mode=decision["mode"],
            reasoning=decision["reasoning"],
            confidence=decision["confidence"],
            decision_time=decision_time,
            consulted_ai=consulted_ai
        )

        self.decision_history.append(human_decision)

        # Track AI agreement
        if consulted_ai and ai_recommendation:
            if human_decision.chosen_mode == ai_recommendation.recommended_mode:
                self.ai_agreement_count += 1

        return human_decision

    def _calculate_complexity(self, shipment: Shipment) -> float:
        """
        Calculate shipment complexity score
        """
        complexity = 0.0

        # Special requirements increase complexity
        complexity += len(shipment.special_requirements) * 0.2

        # High value increases complexity
        if shipment.value > 1000000:
            complexity += 0.3

        # Tight deadline increases complexity
        if shipment.due_date < 10:
            complexity += 0.2

        # Weight affects complexity
        if shipment.weight > 3000:
            complexity += 0.1

        return min(1.0, complexity)

    def _make_reasoned_decision(self, shipment: Shipment,
                              ai_recommendation: Optional[AIRecommendation],
                              ai_influence: float) -> Dict:
        """
        Make reasoned decision based on expertise and AI input
        """
        # Calculate base preferences
        air_preference = 0.5  # Neutral starting point

        # Priority influence
        if shipment.priority == "high":
            air_preference += self.decision_patterns["high_priority_air_preference"] * 0.3
        elif shipment.priority == "low":
            air_preference -= 0.2

        # Cost sensitivity influence
        ocean_cost = shipment.weight * params.costs["ocean"][shipment.destination]
        air_cost = shipment.weight * params.costs["air"][shipment.destination]
        cost_ratio = ocean_cost / air_cost

        if cost_ratio < 0.7:  # Ocean much cheaper
            air_preference -= self.decision_patterns["cost_sensitivity"] * 0.4
        elif cost_ratio > 1.3:  # Air relatively cheaper
            air_preference += 0.2

        # Special requirements influence
        if "temperature_controlled" in shipment.special_requirements:
            if self.expert.specialization == "air_freight":
                air_preference += 0.3
            else:
                air_preference += 0.1

        if "hazardous" in shipment.special_requirements:
            if self.expert.specialization == "risk_management":
                air_preference -= 0.2  # More cautious about hazardous air freight

        # Risk assessment
        if shipment.due_date <= params.transit_times["ocean"][shipment.destination]:
            air_preference += 0.4  # Ocean would miss deadline

        # AI influence
        if ai_recommendation:
            if ai_recommendation.recommended_mode == "air":
                air_preference += ai_influence
            else:
                air_preference -= ai_influence

        # Make final decision
        if air_preference > 0.5:
            chosen_mode = "air"
        else:
            chosen_mode = "ocean"

        # Generate reasoning
        reasoning = self._generate_human_reasoning(shipment, chosen_mode, air_preference)

        # Calculate confidence
        confidence = abs(air_preference - 0.5) * 2  # Convert to 0-1 scale
        confidence = min(0.95, confidence * self.expert.reliability_score)

        return {
            "mode": chosen_mode,
            "reasoning": reasoning,
            "confidence": confidence
        }

    def _generate_human_reasoning(self, shipment: Shipment, chosen_mode: str,
                                  preference_score: float) -> str:
        """
        Generate human-like reasoning for decision
        """
        reasoning_parts = []

        # Start with primary factor
        if shipment.priority == "high" and chosen_mode == "air":
            reasoning_parts.append("High priority requires air transport to meet deadline")
        elif chosen_mode == "ocean" and shipment.priority == "low":
            reasoning_parts.append("Low priority allows cost-effective ocean transport")

        # Add cost consideration
        ocean_cost = shipment.weight * params.costs["ocean"][shipment.destination]
        air_cost = shipment.weight * params.costs["air"][shipment.destination]

        if chosen_mode == "ocean" and ocean_cost < air_cost * 0.8:
            reasoning_parts.append(f"Significant cost savings ({(air_cost - ocean_cost):.0f}) with ocean freight")
        elif chosen_mode == "air" and air_cost <= ocean_cost * 1.2:
            reasoning_parts.append("Air freight cost premium is acceptable for this shipment")

        # Add special requirements
        if "temperature_controlled" in shipment.special_requirements:
            if chosen_mode == "air":
                reasoning_parts.append("Temperature control better maintained with faster air transit")
            else:
                reasoning_parts.append("Ocean refrigeration adequate for this temperature-sensitive cargo")

        # Add expertise-based reasoning
        if self.expert.specialization == "risk_management" and chosen_mode == "ocean":
            reasoning_parts.append("Ocean transport has predictable risk patterns")
        elif self.expert.specialization == "air_freight" and chosen_mode == "air":
            reasoning_parts.append("Air freight offers better control and visibility")

        # Add experience-based judgment
        if preference_score > 0.7:
            reasoning_parts.append("Strong preference based on experience with similar shipments")
        elif preference_score < 0.3:
            reasoning_parts.append("Clear choice based on shipment characteristics")

        return "; ".join(reasoning_parts)

    def get_agreement_rate(self) -> float:
        """
        Calculate AI agreement rate
        """
        if self.ai_consultation_count == 0:
            return 0.0
        return self.ai_agreement_count / self.ai_consultation_count

# Initialize human decision makers
human_decision_makers = [HumanDecisionMaker(expert) for expert in experts]
print("Human Decision Makers initialized:")
for i, hdm in enumerate(human_decision_makers):
    print(f"  {hdm.expert.name}: {hdm.expert.specialization}, {hdm.expert.experience_years} years experience")

Human Decision Makers initialized:
  Sarah Chen: air_freight, 18 years experience
  Marcus Rodriguez: ocean_logistics, 22 years experience
  Aisha Patel: risk_management, 15 years experience


In [ ]:
class SymbioticPartnership:
    """
    Human-AI symbiotic partnership system
    Facilitates collaboration and learning between humans and AI
    """

    def __init__(self, ai_system: ExplainableAI, human_decision_makers: List[HumanDecisionMaker]):
        self.ai_system = ai_system
        self.human_decision_makers = human_decision_makers

        # Partnership metrics
        self.partnership_history = []
        self.trust_scores = {hdm.expert.expert_id: 0.5 for hdm in human_decision_makers}
        self.agreement_rates = {hdm.expert.expert_id: 0.0 for hdm in human_decision_makers}
        self.learning_outcomes = []

        # Partnership development stages
        self.current_stage = "initialization"  # initialization, development, mature, optimized
        self.stage_transitions = []

        # Performance tracking
        self.performance_metrics = {
            "decision_quality": [],
            "decision_time": [],
            "agreement_rate": [],
            "trust_development": [],
            "learning_effectiveness": []
        }

    def make_collaborative_decision(self, shipment: Shipment) -> SymbioticDecision:
        """
        Make collaborative decision through human-AI partnership
        """
        start_time = time.time()

        # AI provides recommendation
        ai_recommendation = self.ai_system.analyze_shipment(shipment)

        # Select appropriate human expert
        human_expert = self._select_human_expert(shipment)

        # Human makes decision with AI consultation
        human_decision = human_expert.make_decision(shipment, ai_recommendation)

        # Determine agreement and learning outcome
        agreement = (human_decision.chosen_mode == ai_recommendation.recommended_mode)
        learning_outcome = self._determine_learning_outcome(
            human_decision, ai_recommendation, agreement
        )

        # Calculate trust score
        trust_score = self._calculate_trust_score(
            human_expert.expert.expert_id, agreement, learning_outcome
        )

        # Create symbiotic decision
        decision_time = int((time.time() - start_time) * 60)

        symbiotic_decision = SymbioticDecision(
            shipment_id=shipment.id,
            final_mode=human_decision.chosen_mode,  # Human has final say
            human_confidence=human_decision.confidence,
            ai_confidence=ai_recommendation.confidence,
            agreement=agreement,
            decision_time=decision_time,
            learning_outcome=learning_outcome,
            trust_score=trust_score
        )

        # Record partnership interaction
        self.partnership_history.append({
            "shipment_id": shipment.id,
            "human_expert": human_expert.expert.expert_id,
            "human_decision": human_decision.chosen_mode,
            "ai_recommendation": ai_recommendation.recommended_mode,
            "agreement": agreement,
            "trust_score": trust_score,
            "learning_outcome": learning_outcome,
            "timestamp": datetime.now()
        })

        # AI learns from human feedback
        self.ai_system.incorporate_feedback(
            shipment.id, human_decision.chosen_mode,
            human_decision.reasoning, agreement
        )

        # Update partnership metrics
        self._update_partnership_metrics(symbiotic_decision)

        # Check for stage transition
        self._check_stage_transition()

        return symbiotic_decision

    def _select_human_expert(self, shipment: Shipment) -> HumanDecisionMaker:
        """
        Select most appropriate human expert for shipment
        """
        expert_scores = []

        for hdm in self.human_decision_makers:
            score = 0.0

            # Specialization match
            if hdm.expert.specialization == "air_freight" and shipment.priority == "high":
                score += 0.3
            elif hdm.expert.specialization == "ocean_logistics" and shipment.priority != "high":
                score += 0.3
            elif hdm.expert.specialization == "risk_management" and shipment.special_requirements:
                score += 0.3

            # Experience weight
            score += hdm.expert.experience_years / 30 * 0.2  # Max 20 years for full score

            # Trust score weight
            score += self.trust_scores[hdm.expert.expert_id] * 0.2

            # Reliability weight
            score += hdm.expert.reliability_score * 0.3

            expert_scores.append(score)

        # Select expert with highest score
        best_expert_idx = np.argmax(expert_scores)
        return self.human_decision_makers[best_expert_idx]

    def _determine_learning_outcome(self, human_decision: HumanDecision,
                                  ai_recommendation: AIRecommendation,
                                  agreement: bool) -> str:
        """
        Determine learning outcome from human-AI interaction
        """
        if agreement:
            return "mutual_agreement"
        else:
            # Determine who was "correct" based on confidence and expertise
            human_weight = human_decision.confidence * human_decision.decision_makers[0].expert.reliability_score
            ai_weight = ai_recommendation.confidence * self.ai_system.model_accuracy

            if human_weight > ai_weight:
                return "human_corrected"
            else:
                return "ai_corrected"

    def _calculate_trust_score(self, expert_id: str, agreement: bool,
                              learning_outcome: str) -> float:
        """
        Calculate and update trust score for human-AI partnership
        """
        current_trust = self.trust_scores[expert_id]

        # Update trust based on interaction
        if agreement:
            trust_change = 0.05  # Increase trust for agreement
        elif learning_outcome == "human_corrected":
            trust_change = 0.02  # Small increase for human expertise
        else:  # ai_corrected
            trust_change = -0.01  # Small decrease for AI correction

        # Apply trust change with bounds
        new_trust = max(0.1, min(0.95, current_trust + trust_change))
        self.trust_scores[expert_id] = new_trust

        return new_trust

    def _update_partnership_metrics(self, decision: SymbioticDecision):
        """
        Update partnership performance metrics
        """
        # Decision quality (simulated based on agreement and confidence)
        quality_score = (decision.human_confidence + decision.ai_confidence) / 2
        if decision.agreement:
            quality_score *= 1.1  # Boost for agreement

        self.performance_metrics["decision_quality"].append(quality_score)
        self.performance_metrics["decision_time"].append(decision.decision_time)
        self.performance_metrics["agreement_rate"].append(1.0 if decision.agreement else 0.0)
        self.performance_metrics["trust_development"].append(decision.trust_score)

        # Learning effectiveness
        learning_score = 0.5
        if decision.learning_outcome == "mutual_agreement":
            learning_score = 0.8
        elif decision.learning_outcome in ["human_corrected", "ai_corrected"]:
            learning_score = 0.6

        self.performance_metrics["learning_effectiveness"].append(learning_score)
        self.learning_outcomes.append(decision.learning_outcome)

    def _check_stage_transition(self):
        """
        Check if partnership should transition to next development stage
        """
        if len(self.partnership_history) < 5:
            return  # Need more history

        # Calculate recent performance
        recent_agreements = sum(1 for p in self.partnership_history[-10:] if p["agreement"])
        agreement_rate = recent_agreements / min(10, len(self.partnership_history))
        avg_trust = np.mean(list(self.trust_scores.values()))

        # Stage transition logic
        if self.current_stage == "initialization":
            if agreement_rate > 0.6 and avg_trust > 0.6:
                self._transition_to_stage("development")
        elif self.current_stage == "development":
            if agreement_rate > 0.8 and avg_trust > 0.75:
                self._transition_to_stage("mature")
        elif self.current_stage == "mature":
            if agreement_rate > 0.9 and avg_trust > 0.85:
                self._transition_to_stage("optimized")

    def _transition_to_stage(self, new_stage: str):
        """
        Transition partnership to new development stage
        """
        self.stage_transitions.append({
            "from_stage": self.current_stage,
            "to_stage": new_stage,
            "timestamp": datetime.now(),
            "decisions_made": len(self.partnership_history)
        })

        self.current_stage = new_stage
        print(f"Partnership transitioned to {new_stage} stage after {len(self.partnership_history)} decisions")

    def run_partnership_simulation(self, shipments: List[Shipment], num_weeks: int = 12) -> Dict:
        """
        Run extended partnership simulation
        """
        print(f"\nStarting Human-AI Partnership Simulation ({num_weeks} weeks)...")
        print("=" * 80)

        simulation_results = {
            "weekly_decisions": [],
            "performance_evolution": [],
            "trust_development": [],
            "learning_patterns": [],
            "stage_progression": []
        }

        decisions_per_week = len(shipments) // num_weeks

        for week in range(num_weeks):
            week_decisions = []
            week_start = len(self.partnership_history)

            # Process weekly shipments
            for i in range(decisions_per_week):
                if week * decisions_per_week + i < len(shipments):
                    shipment = shipments[week * decisions_per_week + i]
                    decision = self.make_collaborative_decision(shipment)
                    week_decisions.append(decision)

            # Calculate weekly metrics
            if week_decisions:
                weekly_metrics = self._calculate_weekly_metrics(week_decisions)

                simulation_results["weekly_decisions"].append(len(week_decisions))
                simulation_results["performance_evolution"].append(weekly_metrics)
                simulation_results["trust_development"].append(dict(self.trust_scores))
                simulation_results["learning_patterns"].append(
                    Counter([d.learning_outcome for d in week_decisions])
                )
                simulation_results["stage_progression"].append(self.current_stage)

                print(f"Week {week+1}: {len(week_decisions)} decisions, "
                      f"Agreement: {weekly_metrics['agreement_rate']:.1%}, "
                      f"Trust: {weekly_metrics['avg_trust']:.2f}, "
                      f"Stage: {self.current_stage}")

        return simulation_results

    def _calculate_weekly_metrics(self, decisions: List[SymbioticDecision]) -> Dict:
        """
        Calculate weekly performance metrics
        """
        agreement_rate = sum(1 for d in decisions if d.agreement) / len(decisions)
        avg_trust = np.mean([d.trust_score for d in decisions])
        avg_decision_time = np.mean([d.decision_time for d in decisions])
        avg_human_confidence = np.mean([d.human_confidence for d in decisions])
        avg_ai_confidence = np.mean([d.ai_confidence for d in decisions])

        learning_distribution = Counter([d.learning_outcome for d in decisions])

        return {
            "agreement_rate": agreement_rate,
            "avg_trust": avg_trust,
            "avg_decision_time": avg_decision_time,
            "avg_human_confidence": avg_human_confidence,
            "avg_ai_confidence": avg_ai_confidence,
            "learning_distribution": learning_distribution
        }

    def generate_partnership_report(self) -> Dict:
        """
        Generate comprehensive partnership report
        """
        if not self.partnership_history:
            return {"error": "No partnership history available"}

        total_decisions = len(self.partnership_history)
        final_agreement_rate = sum(1 for p in self.partnership_history if p["agreement"]) / total_decisions
        final_avg_trust = np.mean(list(self.trust_scores.values()))

        learning_summary = Counter(self.learning_outcomes)

        return {
            "partnership_summary": {
                "total_decisions": total_decisions,
                "final_stage": self.current_stage,
                "stage_transitions": len(self.stage_transitions),
                "final_agreement_rate": final_agreement_rate,
                "final_avg_trust": final_avg_trust
            },
            "performance_metrics": {
                "avg_decision_quality": np.mean(self.performance_metrics["decision_quality"]),
                "avg_decision_time": np.mean(self.performance_metrics["decision_time"]),
                "trust_improvement": final_avg_trust - 0.5,  # Started at 0.5
                "learning_effectiveness": np.mean(self.performance_metrics["learning_effectiveness"])
            },
            "learning_summary": dict(learning_summary),
            "expert_performance": {
                expert_id: {
                    "trust_score": trust,
                    "agreement_rate": hdm.get_agreement_rate()
                }
                for expert_id, trust, hdm in zip(
                    self.trust_scores.keys(),
                    self.trust_scores.values(),
                    self.human_decision_makers
                )
            }
        }

# Initialize symbiotic partnership
partnership = SymbioticPartnership(ai_system, human_decision_makers)
print("\nHuman-AI Symbiotic Partnership initialized:")
print(f"  AI System: Explainable freight optimizer")
print(f"  Human Experts: {len(human_decision_makers)}")
print(f"  Initial Trust Scores: {partnership.trust_scores}")
print(f"  Partnership Stage: {partnership.current_stage}")


Human-AI Symbiotic Partnership initialized:
  AI System: Explainable freight optimizer
  Human Experts: 3
  Initial Trust Scores: {'EXP001': 0.5, 'EXP002': 0.5, 'EXP003': 0.5}
  Partnership Stage: initialization


In [ ]:
try:
    # Run the human-AI partnership simulation
    print("Starting Human-AI Symbiotic Partnership Simulation...")
    print("=" * 80)
    
    # Run initial collaborative decisions
    print("\nInitial Collaborative Decisions:")
    print("-" * 40)
    
    for i, shipment in enumerate(shipments[:5]):  # First 5 shipments
        decision = partnership.make_collaborative_decision(shipment)
    
        print(f"\nShipment {shipment.id} ({shipment.destination}, {shipment.priority} priority):")
        print(f"  Final Decision: {decision.final_mode.upper()}")
        print(f"  Human Confidence: {decision.human_confidence:.1%}")
        print(f"  AI Confidence: {decision.ai_confidence:.1%}")
        print(f"  Agreement: {'YES' if decision.agreement else 'NO'}")
        print(f"  Trust Score: {decision.trust_score:.2f}")
        print(f"  Learning: {decision.learning_outcome}")
        print(f"  Decision Time: {decision.decision_time} minutes")
    
    # Run extended partnership simulation
    simulation_results = partnership.run_partnership_simulation(shipments, num_weeks=8)
    
    # Generate final partnership report
    final_report = partnership.generate_partnership_report()
    
    print(f"\nFinal Partnership Report:")
    print("=" * 80)
    summary = final_report["partnership_summary"]
    metrics = final_report["performance_metrics"]
    
    print(f"Total Decisions: {summary['total_decisions']}")
    print(f"Final Stage: {summary['final_stage']}")
    print(f"Stage Transitions: {summary['stage_transitions']}")
    print(f"Final Agreement Rate: {summary['final_agreement_rate']:.1%}")
    print(f"Final Average Trust: {summary['final_avg_trust']:.2f}")
    
    print(f"\nPerformance Metrics:")
    print(f"  Average Decision Quality: {metrics['avg_decision_quality']:.1%}")
    print(f"  Average Decision Time: {metrics['avg_decision_time']:.1f} minutes")
    print(f"  Trust Improvement: {metrics['trust_improvement']:+.2f}")
    print(f"  Learning Effectiveness: {metrics['learning_effectiveness']:.1%}")
    
    print(f"\nLearning Summary:")
    for outcome, count in final_report["learning_summary"].items():
        percentage = count / sum(final_report["learning_summary"].values()) * 100
        print(f"  {outcome}: {count} ({percentage:.1f}%)")
    
    print(f"\nExpert Performance:")
    for expert_id, performance in final_report["expert_performance"].items():
        expert_name = next(hdm.expert.name for hdm in human_decision_makers
                          if hdm.expert.expert_id == expert_id)
        print(f"  {expert_name}:")
        print(f"    Trust Score: {performance['trust_score']:.2f}")
        print(f"    Agreement Rate: {performance['agreement_rate']:.1%}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: HumanDecision.__init__() got an unexpected keyword argument 'decision_time'...]

In [ ]:
try:
    # Create comprehensive human-AI partnership visualization
    from collections import Counter
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Human-AI Symbiotic Partnership Analysis', fontsize=16, fontweight='bold')
    
    # 1. Trust development over time
    trust_history = simulation_results["trust_development"]
    weeks = range(1, len(trust_history) + 1)
    
    for expert_id in trust_history[0].keys():
        trust_values = [week_data[expert_id] for week_data in trust_history]
        expert_name = next(hdm.expert.name for hdm in human_decision_makers
                          if hdm.expert.expert_id == expert_id)
        ax1.plot(weeks, trust_values, marker='o', linewidth=2, label=expert_name, markersize=4)
    
    ax1.set_xlabel('Week')
    ax1.set_ylabel('Trust Score')
    ax1.set_title('Trust Development Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # 2. Agreement rate and decision quality evolution
    performance_data = simulation_results["performance_evolution"]
    agreement_rates = [p['agreement_rate'] for p in performance_data]
    avg_trust = [p['avg_trust'] for p in performance_data]
    decision_times = [p['avg_decision_time'] for p in performance_data]
    
    ax2.plot(weeks, agreement_rates, 'b-o', linewidth=2, label='Agreement Rate', markersize=4)
    ax2_twin = ax2.twinx()
    ax2_twin.plot(weeks, avg_trust, 'r-s', linewidth=2, label='Average Trust', markersize=4)
    
    ax2.set_xlabel('Week')
    ax2.set_ylabel('Agreement Rate', color='b')
    ax2_twin.set_ylabel('Average Trust', color='r')
    ax2.set_title('Partnership Performance Evolution')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2_twin.tick_params(axis='y', labelcolor='r')
    ax2.legend(loc='upper left')
    ax2_twin.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    # 3. Learning outcomes distribution
    all_learning_outcomes = []
    for week_patterns in simulation_results["learning_patterns"]:
        for outcome, count in week_patterns.items():
            all_learning_outcomes.extend([outcome] * count)
    
    learning_counts = Counter(all_learning_outcomes)
    outcomes = list(learning_counts.keys())
    counts = list(learning_counts.values())
    
    colors = ['lightgreen' if 'mutual' in outcome else 'lightyellow' if 'corrected' in outcome else 'lightcoral'
             for outcome in outcomes]
    
    bars = ax3.bar(outcomes, counts, color=colors, alpha=0.8)
    ax3.set_xlabel('Learning Outcome')
    ax3.set_ylabel('Count')
    ax3.set_title('Learning Outcomes Distribution')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # Add percentage labels
    total_outcomes = sum(counts)
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        percentage = count / total_outcomes * 100
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                 f'{count}\n({percentage:.1f}%)', ha='center', va='bottom', fontweight='bold')
    
    # 4. Partnership stage progression
    stage_progression = simulation_results["stage_progression"]
    stage_colors = {'initialization': 'red', 'development': 'orange', 'mature': 'lightgreen', 'optimized': 'darkgreen'}
    stage_numeric = {'initialization': 1, 'development': 2, 'mature': 3, 'optimized': 4}
    
    stage_numbers = [stage_numeric[stage] for stage in stage_progression]
    stage_colors_list = [stage_colors[stage] for stage in stage_progression]
    
    ax4.scatter(weeks, stage_numbers, c=stage_colors_list, s=100, alpha=0.7)
    ax4.plot(weeks, stage_numbers, 'k--', alpha=0.3)
    
    ax4.set_xlabel('Week')
    ax4.set_ylabel('Partnership Stage')
    ax4.set_title('Partnership Stage Progression')
    ax4.set_yticks([1, 2, 3, 4])
    ax4.set_yticklabels(['Initialization', 'Development', 'Mature', 'Optimized'])
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0.5, 4.5)
    
    # Add stage transition markers
    for i in range(1, len(stage_progression)):
        if stage_progression[i] != stage_progression[i-1]:
            ax4.axvline(x=i+1, color='blue', linestyle=':', alpha=0.5, linewidth=2)
            ax4.text(i+1, 4.3, f'Transition\nto {stage_progression[i]}',
                    ha='center', va='top', fontsize=8, color='blue')
    
    plt.tight_layout()
    plt.show()
    
    print("\nVisualization Summary:")
    print("=" * 80)
    print(f"1. Trust Development: Shows how trust scores evolve for each expert over time")
    print(f"2. Performance Evolution: Agreement rate and average trust progression")
    print(f"3. Learning Distribution: Breakdown of learning outcomes from human-AI interactions")
    print(f"4. Stage Progression: Partnership development stages over the simulation period")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'simulation_results' is not defined...]

In [ ]:
try:
    # Demonstrate explainable AI reasoning and decision transparency
    def demonstrate_explainable_ai(ai_system: ExplainableAI, human_decision_makers: List[HumanDecisionMaker]):
        """
        Demonstrate the explainable AI capabilities and decision transparency
        """
        print("\nDemonstrating Explainable AI Reasoning...")
        print("=" * 80)
    
        # Select a complex shipment for demonstration
        complex_shipment = shipments[3]  # High-value, temperature-controlled to Singapore
    
        print(f"\nAnalyzing Complex Shipment:")
        print(f"  ID: {complex_shipment.id}")
        print(f"  Destination: {complex_shipment.destination}")
        print(f"  Weight: {complex_shipment.weight} kg")
        print(f"  Value: ${complex_shipment.value:,}")
        print(f"  Priority: {complex_shipment.priority}")
        print(f"  Due Date: Day {complex_shipment.due_date}")
        print(f"  Special Requirements: {', '.join(complex_shipment.special_requirements)}")
    
        # Get AI recommendation
        ai_recommendation = ai_system.analyze_shipment(complex_shipment)
    
        print(f"\nAI Recommendation:")
        print(f"  Recommended Mode: {ai_recommendation.recommended_mode.upper()}")
        print(f"  Confidence: {ai_recommendation.confidence:.1%}")
        print(f"  Estimated Cost: ${ai_recommendation.cost_estimate:,.0f}")
        print(f"  Risk Assessment: {ai_recommendation.risk_assessment:.1%}")
    
        print(f"\nAI Reasoning:")
        for i, reason in enumerate(ai_recommendation.reasoning, 1):
            print(f"  {i}. {reason}")
    
        print(f"\nAlternative Options:")
        for alt in ai_recommendation.alternative_modes:
            print(f"  {alt['mode'].upper()}: Score {alt['score']:.2f} - {alt['reason']}")
    
        # Get human decision
        human_expert = human_decision_makers[1]  # Use Marcus Rodriguez (ocean logistics specialist)
        human_decision = human_expert.make_decision(complex_shipment, ai_recommendation)
    
        print(f"\nHuman Expert Decision ({human_expert.expert.name}):")
        print(f"  Chosen Mode: {human_decision.chosen_mode.upper()}")
        print(f"  Confidence: {human_decision.confidence:.1%}")
        print(f"  Decision Time: {human_decision.decision_time} minutes")
        print(f"  Consulted AI: {'Yes' if human_decision.consulted_ai else 'No'}")
        print(f"  Reasoning: {human_decision.reasoning}")
    
        # Analyze agreement/disagreement
        agreement = human_decision.chosen_mode == ai_recommendation.recommended_mode
        print(f"\nCollaboration Analysis:")
        print(f"  Agreement: {'YES' if agreement else 'NO'}")
    
        if agreement:
            print(f"  Outcome: Mutual agreement strengthens partnership")
            print(f"  Learning: Both human and AI reinforce correct decision pattern")
        else:
            print(f"  Outcome: Constructive disagreement enables learning")
            print(f"  Human expertise: {human_expert.expert.specialization} perspective")
            print(f"  AI perspective: Data-driven analytical approach")
            print(f"  Learning opportunity: Cross-validation of decision factors")
    
        return ai_recommendation, human_decision
    
    # Demonstrate trust building and learning
    def demonstrate_trust_building(partnership: SymbioticPartnership):
        """
        Demonstrate trust building and learning mechanisms
        """
        print("\nDemonstrating Trust Building and Learning...")
        print("=" * 80)
    
        # Show trust development trajectory
        print("\nTrust Development Trajectory:")
        for expert_id, final_trust in partnership.trust_scores.items():
            expert_name = next(hdm.expert.name for hdm in partnership.human_decision_makers
                              if hdm.expert.expert_id == expert_id)
            trust_improvement = final_trust - 0.5  # Started at 0.5
            print(f"  {expert_name}: {0.5:.2f} → {final_trust:.2f} ({trust_improvement:+.2f})")
    
        # Show AI learning progress
        print("\nAI Learning Progress:")
        print(f"  Initial Model Accuracy: {0.88:.1%}")
        print(f"  Final Model Accuracy: {partnership.ai_system.model_accuracy:.1%}")
        print(f"  Improvement: {(partnership.ai_system.model_accuracy - 0.88):+.1%}")
        print(f"  Confidence Calibration: {partnership.ai_system.confidence_calibration:.2f}")
    
        # Show learning patterns
        learning_summary = Counter(partnership.learning_outcomes)
        print("\nLearning Patterns Summary:")
        for outcome, count in learning_summary.items():
            percentage = count / sum(learning_summary.values()) * 100
            print(f"  {outcome}: {count} instances ({percentage:.1f}%)")
    
        # Show stage progression
        print("\nPartnership Stage Progression:")
        for i, transition in enumerate(partnership.stage_transitions, 1):
            print(f"  Transition {i}: {transition['from_stage']} → {transition['to_stage']} "
                  f"(after {transition['decisions_made']} decisions)")
    
    # Run demonstrations
    ai_rec, human_dec = demonstrate_explainable_ai(ai_system, human_decision_makers)
    demonstrate_trust_building(partnership)
    
    # Create decision transparency visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Explainable AI and Decision Transparency Analysis', fontsize=16, fontweight='bold')
    
    # 1. AI confidence vs human confidence comparison
    confidences = [(d.ai_confidence, d.human_confidence) for d in partnership.partnership_history]
    ai_confidences = [c[0] for c in confidences]
    human_confidences = [c[1] for c in confidences]
    
    decisions = range(1, len(confidences) + 1)
    ax1.plot(decisions, ai_confidences, 'b-o', linewidth=2, label='AI Confidence', markersize=3, alpha=0.7)
    ax1.plot(decisions, human_confidences, 'r-s', linewidth=2, label='Human Confidence', markersize=3, alpha=0.7)
    
    # Highlight agreements and disagreements
    for i, decision in enumerate(partnership.partnership_history):
        if decision['agreement']:
            ax1.scatter(i+1, confidences[i][0], color='green', s=50, alpha=0.5, marker='v')
            ax1.scatter(i+1, confidences[i][1], color='green', s=50, alpha=0.5, marker='v')
        else:
            ax1.scatter(i+1, confidences[i][0], color='red', s=50, alpha=0.5, marker='x')
            ax1.scatter(i+1, confidences[i][1], color='red', s=50, alpha=0.5, marker='x')
    
    ax1.set_xlabel('Decision Number')
    ax1.set_ylabel('Confidence Level')
    ax1.set_title('AI vs Human Confidence Levels')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # 2. Decision time analysis
    decision_times = [d.decision_time for d in partnership.partnership_history]
    agreement_times = [d.decision_time for d in partnership.partnership_history if d.agreement]
    disagreement_times = [d.decision_time for d in partnership.partnership_history if not d.agreement]
    
    ax2.hist(decision_times, bins=15, alpha=0.5, color='gray', label='All Decisions')
    ax2.hist(agreement_times, bins=10, alpha=0.7, color='green', label='Agreements')
    ax2.hist(disagreement_times, bins=10, alpha=0.7, color='red', label='Disagreements')
    
    ax2.set_xlabel('Decision Time (minutes)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Decision Time Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. AI reasoning factors frequency
    all_reasoning = []
    for rec in ai_system.decision_history:
        all_reasoning.extend(rec.reasoning)
    
    # Extract key reasoning themes
    reasoning_themes = {
        'Cost Consideration': 0,
        'Time Sensitivity': 0,
        'Priority Based': 0,
        'Special Requirements': 0,
        'Value Based': 0
    }
    
    for reason in all_reasoning:
        if 'saves' in reason or 'cost' in reason.lower():
            reasoning_themes['Cost Consideration'] += 1
        if 'days' in reason or 'deadline' in reason.lower() or 'buffer' in reason.lower():
            reasoning_themes['Time Sensitivity'] += 1
        if 'priority' in reason.lower():
            reasoning_themes['Priority Based'] += 1
        if 'temperature' in reason.lower() or 'hazardous' in reason.lower() or 'special' in reason.lower():
            reasoning_themes['Special Requirements'] += 1
        if 'value' in reason.lower() or 'high-value' in reason.lower():
            reasoning_themes['Value Based'] += 1
    
    themes = list(reasoning_themes.keys())
    counts = list(reasoning_themes.values())
    
    bars = ax3.bar(themes, counts, color='lightblue', alpha=0.8)
    ax3.set_xlabel('Reasoning Theme')
    ax3.set_ylabel('Frequency')
    ax3.set_title('AI Reasoning Themes Distribution')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # Add frequency labels
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                 str(count), ha='center', va='bottom', fontweight='bold')
    
    # 4. Trust vs Agreement correlation
    trust_values = [p['trust_score'] for p in partnership.partnership_history]
    agreement_binary = [1 if p['agreement'] else 0 for p in partnership.partnership_history]
    
    # Create scatter plot with trend line
    ax4.scatter(trust_values, agreement_binary, alpha=0.6, s=50)
    
    # Add trend line
    z = np.polyfit(trust_values, agreement_binary, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(min(trust_values), max(trust_values), 100)
    ax4.plot(x_trend, p(x_trend), 'r--', alpha=0.8, linewidth=2)
    
    ax4.set_xlabel('Trust Score')
    ax4.set_ylabel('Agreement (1=Yes, 0=No)')
    ax4.set_title('Trust Score vs Agreement Correlation')
    ax4.set_ylim(-0.1, 1.1)
    ax4.grid(True, alpha=0.3)
    
    # Add correlation coefficient
    correlation = np.corrcoef(trust_values, agreement_binary)[0, 1]
    ax4.text(0.05, 0.95, f'Correlation: {correlation:.3f}', transform=ax4.transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nExplainable AI Analysis Summary:")
    print("=" * 80)
    print(f"1. Confidence Analysis: AI and human confidence levels tracked over time")
    print(f"2. Decision Time: Agreement decisions typically faster than disagreements")
    print(f"3. Reasoning Themes: Cost and time sensitivity are primary AI decision factors")
    print(f"4. Trust-Agreement Correlation: {correlation:.3f} correlation between trust and agreement")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: HumanDecision.__init__() got an unexpected keyword argument 'decision_time'...]

### Why this Tier exists vs earlier Tiers
The Human-AI Symbiotic Partnership represents the pinnacle of decision-making sophistication, combining the best of both worlds:
- **Human wisdom**: Contextual understanding, experience-based intuition, ethical judgment
- **AI analytics**: Data-driven insights, pattern recognition, computational optimization
- **Collaborative intelligence**: Emergent capabilities that neither human nor AI possess alone
- **Adaptive learning**: System improves through continuous feedback and experience
- **Trust development**: Building reliable partnerships through transparent interactions
- **Explainable decisions**: Full transparency in reasoning processes for accountability

### Pros / Cons vs Tiers 1-5
**Pros:**
- Superior decision quality through human-AI synergy
- Continuous learning and improvement over time
- Explainable reasoning for regulatory compliance and stakeholder trust
- Adaptive to changing conditions and new patterns
- Risk mitigation through dual validation
- Knowledge transfer between human expertise and AI capabilities
- Scalable expertise through AI assistance

**Cons:**
- High implementation complexity and integration requirements
- Significant training and change management needs
- Dependence on human expert availability and engagement
- Longer initial deployment time before reaching optimal performance
- Requires ongoing maintenance and calibration
- Potential for human-AI conflicts requiring resolution mechanisms
- Higher costs due to dual system maintenance

### When to use this Tier
- **Critical decisions** where both data analytics and human wisdom are essential
- **Regulated industries** requiring explainable decisions and human accountability
- **High-stakes environments** where errors have significant consequences
- **Knowledge transfer** situations where expert expertise needs to be scaled
- **Dynamic environments** requiring continuous adaptation and learning
- **Complex problems** with multiple competing objectives and constraints
- **Strategic decisions** involving ethical considerations and long-term impacts
- **Customer-facing operations** requiring transparency and trust building